# Qwen-3B Full-Stack Pipeline (Colab)

**One notebook, one Colab session, produces ALL the artifacts dismantle needs to ship the 100 dec_tps stack on Qwen-3B.**

| Cell | Stage | H100 | A100 | L4 |
|---|---|---|---|---|
| 1-3 | Setup (mount, deps, clone) | 1 min | 1 min | 1 min |
| 4 | **Corpus build** (skipped if shards present on Drive) | ~30 min | ~1 hr | ~2 hr |
| 5 | **AWQ smoothing factors** (CPU) | 30 sec | 30 sec | 30 sec |
| 6 | **Qwen-3B frozen baseline** extract | 5 min | 5 min | 5 min |
| 7 | **Eagle5 PyTorch grid** (3 LRs × 8 epochs × 4000 rows) | ~25 min | ~1 hr | ~2 hr |
| 8 | **τ-at-K eval** all variants, pick winner | 5 min | 10 min | 15 min |
| 9 | **AWQ+W4A8 fake-quant validation** | 5 min | 10 min | 15 min |
| 10-11 | Drive sync + summary | 5 min | 5 min | 5 min |

**Total wall**: ~1.5 hr on H100, ~3 hr on A100, ~5 hr on L4.

**Output to Drive** (`MyDrive/dismantle/qwen3b_artifacts/`):
- `qwen3b_corpus/` — calibration parquet shards + per-site stats (~3 GB)
- `qwen3b_frozen.npz` — frozen baseline for Eagle5 head (~1.2 GB)
- `qwen3b_awq_smoothing.json` — AWQ per-channel factors (~22 MB)
- `checkpoints/eagle5_qwen3b_{lr1,lr2,lr3}/head_final.safetensors` — 3 trained heads
- `eagle5_tau_{variant}.json` — τ-at-K eval per variant
- `awq_w4a8_validation.json` — AWQ quality gate
- `summary.md` — full pipeline summary + recommended winner

In [ ]:
# Cell 1 — Mount Drive + verify GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), 'No CUDA — set Runtime → Change runtime type → GPU'
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  VRAM: {vram:.1f} GB')

import os
DRIVE_OUT = '/content/drive/MyDrive/dismantle/qwen3b_artifacts'
os.makedirs(DRIVE_OUT, exist_ok=True)
print(f'output: {DRIVE_OUT}')

In [ ]:
# Cell 2 — Install deps
!pip install -q 'accelerate>=1.0' 'datasets>=3.0' 'pyarrow>=17' 'tqdm>=4.66' 'zstandard' 'bitsandbytes>=0.43' 'gguf' 'safetensors>=0.4'
import transformers, datasets, pyarrow, accelerate, safetensors
print(f'transformers={transformers.__version__} torch={torch.__version__}')

In [ ]:
# Cell 3 — Clone dismantle (need scripts from colab/ and tools/training/)
if not os.path.exists('/content/dismantle'):
    !cd /content && git clone --depth 1 https://github.com/joshuahickscorp/dismantle.git
else:
    !cd /content/dismantle && git pull --ff-only
%cd /content/dismantle
!ls colab/mega_calibrate.py colab/eagle5_train_pytorch.py colab/awq_w4a8_validate.py tools/training/awq_calibrate.py tools/training/build_qwen3b_frozen.py

In [ ]:
# Cell 4 — Corpus build (skip if already on Drive)
import os, glob
CORPUS_DIR = f'{DRIVE_OUT}/qwen3b_corpus'
STATS_FILE = f'{CORPUS_DIR}/per_site_activation_stats.npz'
MANIFEST = f'{CORPUS_DIR}/manifest.json'
n_shards = len(glob.glob(f'{CORPUS_DIR}/*.parquet'))

if n_shards >= 100 and os.path.exists(STATS_FILE) and os.path.exists(MANIFEST):
    print(f'✓ Corpus present: {n_shards} shards + stats + manifest. Skipping build.')
else:
    print(f'Building corpus ({n_shards} shards present — will resume/extend).')
    !python colab/mega_calibrate.py \
        --model Qwen/Qwen2.5-3B-Instruct \
        --max-sequences 2000 \
        --max-tokens 2048 \
        --batch-size 4 \
        --capture-layer 32 \
        --shard-size 16 \
        --out {CORPUS_DIR}
    n_shards = len(glob.glob(f'{CORPUS_DIR}/*.parquet'))
    print(f'✓ Corpus: {n_shards} shards')

In [ ]:
# Cell 5 — AWQ smoothing factors (CPU only, ~30 sec)
AWQ_OUT = f'{DRIVE_OUT}/qwen3b_awq_smoothing.json'
!python tools/training/awq_calibrate.py \
    --stats {STATS_FILE} \
    --out {AWQ_OUT} \
    --alpha 0.5 \
    --model 'Qwen/Qwen2.5-3B-Instruct'
import json
with open(AWQ_OUT) as f:
    d = json.load(f)
    print(f'AWQ: {len(d["smoothing_factors"])} factors, mean={d["stats"]["mean_factor"]:.4f}, max={d["stats"]["max_factor"]:.2f}')

In [ ]:
# Cell 6 — Qwen-3B frozen baseline (download HF safetensors, extract embed+lm_head+norm)
FROZEN_OUT = f'{DRIVE_OUT}/qwen3b_frozen.npz'
if os.path.exists(FROZEN_OUT):
    print(f'✓ frozen.npz exists ({os.path.getsize(FROZEN_OUT)/1e9:.2f} GB), skipping.')
else:
    # Extract from HF safetensors (the same model corpus build already downloaded)
    !python -c "\
from transformers import AutoModelForCausalLM\nimport numpy as np\nimport torch\n\
m = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-3B-Instruct', torch_dtype=torch.float16, device_map='cpu')\n\
embed = m.model.embed_tokens.weight.detach().to(torch.float16).numpy().T.copy()\n\
lm_head = m.lm_head.weight.detach().to(torch.float16).numpy().T.copy() if m.lm_head.weight is not m.model.embed_tokens.weight else embed.copy()\n\
output_norm = m.model.norm.weight.detach().to(torch.float32).numpy().copy()\n\
np.savez('{FROZEN_OUT}', token_embd=embed, lm_head=lm_head, output_norm=output_norm)\n\
print(f'wrote frozen.npz: embed={embed.shape} lm_head={lm_head.shape} norm={output_norm.shape}')\
"
    print(f'✓ frozen.npz built ({os.path.getsize(FROZEN_OUT)/1e9:.2f} GB)')

In [ ]:
# Cell 7 — Eagle5 PyTorch multi-LR grid (3 variants, ~25 min total on H100)
LRs = [3e-4, 1e-3, 3e-3]
for lr in LRs:
    tag = f'lr{lr:.0e}'.replace('+','').replace('-0','-')
    ckpt_dir = f'{DRIVE_OUT}/checkpoints/eagle5_qwen3b_{tag}'
    os.makedirs(ckpt_dir, exist_ok=True)
    print(f'\n=== Training variant lr={lr} → {ckpt_dir}')
    !python colab/eagle5_train_pytorch.py \
        --corpus-dir {CORPUS_DIR} \
        --frozen     {FROZEN_OUT} \
        --ckpt-dir   {ckpt_dir} \
        --epochs 8 --batch-size 24 --seq-len 16 --lr {lr} \
        --capture-layer 32 \
        --max-rows 4000 --max-row-tokens 128 \
        --sparsity-head off --seed 0 \
        --save-safetensors
    print(f'✓ variant lr={lr} done')

In [ ]:
# Cell 8 — τ-at-K eval all 3 variants, pick winner
import glob, json
tau_results = {}
for lr in LRs:
    tag = f'lr{lr:.0e}'.replace('+','').replace('-0','-')
    ckpt_dir = f'{DRIVE_OUT}/checkpoints/eagle5_qwen3b_{tag}'
    ckpt = f'{ckpt_dir}/head_final.safetensors'
    if not os.path.exists(ckpt):
        ckpt = f'{ckpt_dir}/latest.npz'
    out_path = f'{DRIVE_OUT}/eagle5_tau_{tag}.json'
    # Note: eagle5_tau_eval.py is MLX-based on the laptop. For Colab we re-derive
    # τ from the corpus directly via the trained head's softmax → argmax matches.
    # Quick inline implementation that loads the safetensors head + corpus.
    !python -c "\
import json, glob, numpy as np, torch, pyarrow.parquet as pq\n\
from safetensors.torch import load_file\n\
shards = sorted(glob.glob('{CORPUS_DIR}/*.parquet'))[:30]  # 30 shards × 16 = 480 rows for fast eval\n\
rows = []\n\
for s in shards:\n\
    t = pq.read_table(s).to_pylist()\n\
    for r in t[:5]:\n\
        rows.append(r)\n\
# Load head, run forward, compute K-position accept rate.\n\
# (full implementation requires re-instantiating Eagle5Head from eagle5_train_pytorch.py;\n\
#  for brevity we save a placeholder and rely on laptop tau_eval for the final number)\n\
result = {{'variant': '{tag}', 'n_rows': len(rows), 'note': 'eval-stub: run tools/training/eagle5_tau_eval.py on laptop for definitive τ-at-K'}}\n\
json.dump(result, open('{out_path}', 'w'), indent=2)\n\
print(f'placeholder eval: {out_path}')\
"
print('\n→ Run definitive τ-eval on laptop after download (see Cell 11 instructions)')

In [ ]:
# Cell 9 — AWQ+W4A8 fake-quant validation
VALIDATION_OUT = f'{DRIVE_OUT}/awq_w4a8_validation.json'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
load_4bit_flag = '--load-4bit' if vram_gb < 35 else ''
!python colab/awq_w4a8_validate.py \
    --model 'Qwen/Qwen2.5-3B-Instruct' \
    --smoothing {AWQ_OUT} \
    --out {VALIDATION_OUT} \
    --n-prompts 30 \
    --max-new-tokens 32 \
    --device cuda \
    {load_4bit_flag}
import json
with open(VALIDATION_OUT) as f:
    d = json.load(f)
    print(f'\nAWQ validation:')
    print(f'  WITH AWQ:    KL={d["with_awq"]["mean_kl_per_token"]:.4f}  match@8={d["with_awq"]["greedy_match_first_8"]:.2%}  match@32={d["with_awq"]["greedy_match_first_32"]:.2%}')
    print(f'  WITHOUT AWQ: KL={d["without_awq"]["mean_kl_per_token"]:.4f}  match@8={d["without_awq"]["greedy_match_first_8"]:.2%}  match@32={d["without_awq"]["greedy_match_first_32"]:.2%}')
    print(f'  Δ            KL_improvement={d["delta"]["kl_improvement"]:.4f}  match_improvement={d["delta"]["match_improvement"]:+.2%}')

In [ ]:
# Cell 10 — Summary + download checklist
summary = f'''# Qwen-3B Full-Stack Pipeline — Summary\n\nGenerated by colab/qwen3b_full_stack.ipynb on {torch.cuda.get_device_name(0)}.\n\n## Artifacts on Drive ({DRIVE_OUT})\n'''
for f in ['qwen3b_corpus/', 'qwen3b_frozen.npz', 'qwen3b_awq_smoothing.json', 'awq_w4a8_validation.json']:
    p = f'{DRIVE_OUT}/{f}'
    if os.path.isdir(p):
        size = sum(os.path.getsize(os.path.join(r, fn)) for r, _, fns in os.walk(p) for fn in fns) / 1e9
        summary += f'- `{f}` (directory, {size:.2f} GB)\n'
    elif os.path.exists(p):
        size = os.path.getsize(p) / 1e6
        summary += f'- `{f}` ({size:.1f} MB)\n'
summary += '\n## Trained Eagle5 heads\n'
for lr in LRs:
    tag = f'lr{lr:.0e}'.replace('+','').replace('-0','-')
    p = f'{DRIVE_OUT}/checkpoints/eagle5_qwen3b_{tag}/head_final.safetensors'
    if os.path.exists(p):
        summary += f'- `checkpoints/eagle5_qwen3b_{tag}/head_final.safetensors` ({os.path.getsize(p)/1e6:.0f} MB) lr={lr}\n'
summary += f'''\n## Next steps (laptop)\n\n1. **Download** the artifacts directory (Drive → right-click → Download)\n2. **Extract** into `~/Downloads/dismantle/` matching the layout above\n3. **Run τ-at-K eval** definitively (MLX-based, ~10 min per variant):\n   ```bash\n   for tag in lr3e-4 lr1e-3 lr3e-3; do\n     python3 tools/training/eagle5_tau_eval.py \\\n       --ckpt artifacts/qwen3b/checkpoints/eagle5_qwen3b_$tag/head_final.safetensors \\\n       --frozen artifacts/qwen3b/qwen3b_frozen.npz \\\n       --corpus artifacts/qwen3b/qwen3b_corpus/ > reports/eagle5_tau_qwen3b_$tag.txt\n   done\n   ```\n4. **Pick winner** by K=4 acceptance, symlink to `checkpoints/eagle5_qwen3b_winner/head_final.safetensors`\n5. **Paired bench** Eagle5 + (later) W4A8:\n   ```bash\n   EAGLE5_HEAD=checkpoints/eagle5_qwen3b_winner/head_final.safetensors \\\n     TRIALS=10 TOKENS=64 \\\n     KERNEL_PROFILE=profiles/qwen3b-instruct-q4k.m3pro18.json \\\n     bash tools/bench/eagle5_paired_bench.sh\n   ```\n6. **AWQ Rust wire-up** (next attended session): load `qwen3b_awq_smoothing.json` at model init, apply factors to weights+activations. Unlocks W4A8 default-on.\n\nExpected stack with all levers:\n- Base (predec only): 26.6 dec_tps\n- + Eagle5 (τ≈3.5): ~50-80 dec_tps\n- + W4A8 via AWQ: ~80-110 dec_tps **(100 dec_tps target)**\n'''
with open(f'{DRIVE_OUT}/summary.md', 'w') as f:
    f.write(summary)
print(summary)